# Gradient x Activation Attribution

### Overview

Computes a gradient-based attribution score, a cheap proxy for activation patching, to measure how much each token position contributes to the model's prediction. This notebook is an unexecuted scaffold — the routine is defined but never run on data.

### Setup

In [ ]:
from typing import List
import torch
import gc
import random
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("src")
import _util
import _dataset
import _prompt
import _mapping
from _intervention import forward_with_cache

### Loss Function

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

## Attribution Routine

For each prompt, caches an early-layer activation, backpropagates the loss on the label token, and scores each position by activation times gradient.

In [ ]:
def grad_embedding_attribution(model,
                          tokenizer,
                          clean_prompts: List[str],
                          labels: List[str],
                          ) -> None:

    """
    Calculates gradient-based attribution scores for token embeddings.

    This function computes attribution scores by:
    1. Running forward pass through the model
    2. Computing gradients 
    3. Calculating attribution scores as norm of (activation * gradient) for each position

    Args:
        model (str): Name/path of the model to load
        exp (Experiment): Experiment configuration object
        clean_prompts (List[str]): List of input prompts to analyze
        correct_tokens (List[str]): List of correct answer tokens
        wrong_tokens (List[str]): List of incorrect answer tokens

    Returns:
        Tuple[List[List[float]], List[List[str]]]: Returns two lists:
            - List of attribution scores for each position in each prompt
            - List of tokenized prompts
    """
        
    attribute_scores = []
    prompts = []
    for index, example in enumerate(clean_prompts):
        tokens = tokenizer(example, add_special_tokens=True, return_tensors="pt")["input_ids"]
        output, activations = forward_with_cache(model, tokens)
        input_acts = activations['model.layers.0']
        
        label_tokens = tokenizer(labels[index], add_special_tokens=False, return_tensors="pt")["input_ids"]
        
        loss = loss_fn(output.logits[:,-1,:], label_tokens[:,0])
        loss.backward()
        input_grad = model.model.layers[0].grad

        score = torch.norm(input_acts * input_grad, dim=-1).tolist()
        
        attribute_scores.append(score)
        del clean_resid_pre_act, grad_resid, score
    return attribute_scores, prompts